##  TechMind — Exploración y Preparación del Dataset **StackExchange**

#### Equipo tejONEs

#### 04_exploracion_dataset_stackexchange.ipynb

💡**Dataset**: [StackExchange - Extracción a traves de la API Oficial](https://api.stackexchange.com/)

- El proceso general que sigue este pipeline es:
    - [x]  Extracción de preguntas por términos técnicos desde la API y deduplicación por identificador
    - [x]  Normalización a las siete categorías del proyecto y asignación del tipo de contenido `apunte`
    - [x]  Persistencia del dataset crudo para trazabilidad
    - [x]  Limpieza de título y texto (HTML, URLs, duplicados y filtro de mínimo 100 palabras)
    - [x]  Preselección con margen para traducción y balanceo a 50 registros por categoría
    - [x]  Traducción al español de título y texto, procesamiento NLP y respaldo de traducciones
    - [x]  Validación del esquema y distribución, y exportación del dataset final en `procesados/`

## Importaciones

In [1]:
import os
import re
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd
import requests
import time
from deep_translator import GoogleTranslator
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

# Descargar recursos de NLP
nltk.download('stopwords', quiet=True)
spanish_stopwords = set(stopwords.words('spanish'))

## Configuración

In [2]:
def find_project_root(start: Path) -> Path:
    """Encuentra la raíz del proyecto desde el directorio actual o sus padres."""
    for candidate in [start, *start.parents]:
        if (candidate / 'data_science').exists() and (candidate / 'README.md').exists():
            return candidate
    return start


base_dir = Path.cwd().resolve()
project_root = find_project_root(base_dir)

CARPETA_DATA = str((project_root / 'data_science' / 'data').resolve())
CARPETA_CRUDOS = str((project_root / 'data_science' / 'data' / 'crudos').resolve())
CARPETA_PROCESADOS = str((project_root / 'data_science' / 'data' / 'procesados').resolve())

print(f'✅ Ruta de procesados existe: {Path(CARPETA_PROCESADOS).exists()}')
print(f'✅ Ruta de crudos existe: {Path(CARPETA_CRUDOS).exists()}')
print(f'✅ Ruta de datos existe: {Path(CARPETA_DATA).exists()}')
print(f'📂 Datos procesados: {CARPETA_PROCESADOS}')
print(f'📂 Datos crudos: {CARPETA_CRUDOS}')
print(f'📁 Proyecto local: {CARPETA_DATA}')

✅ Ruta de procesados existe: True
✅ Ruta de crudos existe: True
✅ Ruta de datos existe: True
📂 Datos procesados: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados
📂 Datos crudos: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos
📁 Proyecto local: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data


# 1.Stack Overflow Questions dataset

## 1.1 Prueba de conexión con la API

In [3]:
respuesta = requests.get(
    "https://api.stackexchange.com/2.3/questions",
    params={
        "tagged": "android",
        "site": "stackoverflow",
        "pagesize": 5
    }
)

print("Código de respuesta:", respuesta.status_code)
datos = respuesta.json()
print(datos["items"][0]["title"])

Código de respuesta: 200
com.google.android.gms.common.api.ApiException: 10:


## 1.2 Funciones de extracción y normalización

In [4]:
def buscar_preguntas(tag, cantidad_paginas=1, pagesize=100):
    """
    Trae preguntas de Stack Overflow que tengan un tag específico (ej: 'android').
    Reintenta automáticamente si algo falla, hasta 3 veces por página.
    """
    todas_las_preguntas = []
    url = "https://api.stackexchange.com/2.3/questions"

    for pagina in range(1, cantidad_paginas + 1):
        parametros = {
            "tagged": tag,
            "site": "stackoverflow",
            "pagesize": pagesize,
            "page": pagina,
            "filter": "withbody",   # para que nos traiga el texto completo, no solo el título
            "sort": "votes",
            "order": "desc"
        }

        intentos = 0
        maximo_intentos = 3
        exito = False

        while intentos < maximo_intentos and not exito:
            try:
                respuesta = requests.get(url, params=parametros, timeout=10)

                if respuesta.status_code == 200:
                    datos = respuesta.json()
                    todas_las_preguntas.extend(datos.get("items", []))
                    exito = True
                    print(f"Tag '{tag}', página {pagina}: {len(datos.get('items', []))} preguntas traídas.")

                elif respuesta.status_code == 429:
                    print("La API dice que vamos muy rápido. Esperando 30 segundos...")
                    time.sleep(30)
                    intentos += 1

                else:
                    print(f"Código inesperado ({respuesta.status_code}). Reintentando en 5 segundos...")
                    intentos += 1
                    time.sleep(5)

            except requests.exceptions.RequestException as error:
                print(f"Problema de conexión: {error}. Reintentando en 5 segundos...")
                intentos += 1
                time.sleep(5)

        if not exito:
            print(f"No se pudo traer la página {pagina} del tag '{tag}' después de {maximo_intentos} intentos.")

        time.sleep(1)  # pausa chica entre páginas, para no saturar la API

    return todas_las_preguntas

In [5]:
def buscar_preguntas_por_consulta(consulta, cantidad_paginas=2, pagesize=100):
    """Busca preguntas por texto libre usando el endpoint search/advanced."""
    todas_las_preguntas = []
    url = 'https://api.stackexchange.com/2.3/search/advanced'

    for pagina in range(1, cantidad_paginas + 1):
        parametros = {
            'q': consulta,
            'site': 'stackoverflow',
            'pagesize': pagesize,
            'page': pagina,
            'filter': 'withbody',
            'sort': 'votes',
            'order': 'desc',
        }

        for intento in range(1, 4):
            try:
                respuesta = requests.get(url, params=parametros, timeout=30)
                respuesta.raise_for_status()
                datos = respuesta.json()
                items = datos.get('items', [])
                todas_las_preguntas.extend(items)
                print(f"Consulta '{consulta}', página {pagina}: {len(items)} preguntas traídas.")
                if datos.get('backoff'):
                    time.sleep(datos['backoff'])
                break
            except requests.exceptions.RequestException as error:
                print(f"Intento {intento}/3 falló para '{consulta}': {error}")
                if intento < 3:
                    time.sleep(5)
        else:
            print(f"No se pudo traer la página {pagina} para la consulta '{consulta}'.")

        time.sleep(1)

    return todas_las_preguntas

In [6]:
preguntas_android = buscar_preguntas(tag="android", cantidad_paginas=1, pagesize=20)
print("\nTotal de preguntas traídas:", len(preguntas_android))
print("Primer título:", preguntas_android[0]["title"])

Tag 'android', página 1: 20 preguntas traídas.

Total de preguntas traídas: 20
Primer título: What is the difference between px, dip, dp, and sp?


In [7]:
def convertir_a_dataframe(preguntas, tag_usado):
    filas = []
    for pregunta in preguntas:
        filas.append({
            "id_pregunta": pregunta.get("question_id"),
            "titulo": pregunta.get("title", ""),
            "texto": pregunta.get("body", ""),
            "tag_original": pregunta.get("_consulta_busqueda", tag_usado),
            "autor": pregunta.get("owner", {}).get("display_name", "Desconocido"),
            "url": pregunta.get("link", ""),
            "votos": pregunta.get("score", 0)
        })
    return pd.DataFrame(filas)

In [8]:
df_android = convertir_a_dataframe(preguntas_android, "android")
df_android.head()

,id_pregunta,titulo,texto,tag_original,autor,url,votos
0,2025282,"What is the difference between px, dip, dp, an...",<p>What is the difference between the units of...,android,capecrawler,https://stackoverflow.com/questions/2025282/wh...,6432
1,1109022,How can I close/hide the Android soft keyboard...,<p>I have an <code>EditText</code> and a <code...,android,Vidar Vestnes,https://stackoverflow.com/questions/1109022/ho...,4372
2,13375357,Proper use cases for Android UserManager.isUse...,<p>I was looking at the new APIs introduced in...,android,Ovidiu Latcu,https://stackoverflow.com/questions/13375357/p...,4056
3,1554099,Why is the Android emulator so slow? How can w...,<p>I have got a <strong>2.67</strong> GHz Cel...,android,Andrie,https://stackoverflow.com/questions/1554099/wh...,3569
4,1555109,How can I stop EditText from gaining focus whe...,"<p>I have an <code>Activity</code> in Android,...",android,Mark,https://stackoverflow.com/questions/1555109/ho...,3181


## 1.3 Extracción de las siete categorías

In [9]:
preguntas_android = buscar_preguntas(tag="android", cantidad_paginas=2, pagesize=100)
print("\nTotal de preguntas traídas:", len(preguntas_android))

Tag 'android', página 1: 100 preguntas traídas.
Tag 'android', página 2: 100 preguntas traídas.

Total de preguntas traídas: 200


In [10]:
df_android = convertir_a_dataframe(preguntas_android, "android")
print("Filas en la tabla:", len(df_android))
df_android.head()

Filas en la tabla: 200


,id_pregunta,titulo,texto,tag_original,autor,url,votos
0,2025282,"What is the difference between px, dip, dp, an...",<p>What is the difference between the units of...,android,capecrawler,https://stackoverflow.com/questions/2025282/wh...,6432
1,1109022,How can I close/hide the Android soft keyboard...,<p>I have an <code>EditText</code> and a <code...,android,Vidar Vestnes,https://stackoverflow.com/questions/1109022/ho...,4372
2,13375357,Proper use cases for Android UserManager.isUse...,<p>I was looking at the new APIs introduced in...,android,Ovidiu Latcu,https://stackoverflow.com/questions/13375357/p...,4056
3,1554099,Why is the Android emulator so slow? How can w...,<p>I have got a <strong>2.67</strong> GHz Cel...,android,Andrie,https://stackoverflow.com/questions/1554099/wh...,3569
4,1555109,How can I stop EditText from gaining focus whe...,"<p>I have an <code>Activity</code> in Android,...",android,Mark,https://stackoverflow.com/questions/1555109/ho...,3181


In [11]:
CATEGORY_KEYWORDS = {
    'Backend': ['backend', 'server-side', 'api', 'microservice', 'rest api', 'node.js', 'spring boot', '.net', 'java ee'],
    'Frontend': ['frontend', 'front-end', 'react', 'angular', 'vue', 'javascript', 'css', 'user interface', 'single page application'],
    'Data Science': ['data science', 'machine learning', 'deep learning', 'artificial intelligence', 'data analytics', 'neural network', 'data mining', 'big data'],
    'Cloud': ['cloud computing', 'cloud infrastructure', 'aws', 'azure', 'google cloud', 'serverless', 'cloud native', 'saas', 'paas', 'iaas'],
    'DevOps': ['devops', 'continuous integration', 'continuous deployment', 'ci/cd', 'kubernetes', 'docker', 'container orchestration', 'infrastructure as code'],
    'Bases de Datos': ['database', 'sql', 'nosql', 'relational database', 'data warehouse', 'postgresql', 'mongodb', 'database management'],
    'Mobile': ['mobile application', 'android', 'ios', 'mobile computing', 'flutter', 'react native', 'mobile app development', 'smartphone'],
}

TARGET_PER_CATEGORY_STACKEXCHANGE = 50
RAW_TARGET_PER_CATEGORY_STACKEXCHANGE = 250

In [12]:
todos_los_dataframes = []
ids_globales = set()

for categoria, consultas in CATEGORY_KEYWORDS.items():
    print(f"\n=== Extrayendo categoría: {categoria} ===")
    preguntas_categoria = {}

    for consulta in consultas:
        preguntas = buscar_preguntas_por_consulta(consulta, cantidad_paginas=2, pagesize=100)

        for pregunta in preguntas:
            question_id = pregunta.get('question_id') or pregunta.get('link')
            if question_id and question_id not in ids_globales and question_id not in preguntas_categoria:
                registro = dict(pregunta)
                registro['_consulta_busqueda'] = consulta
                preguntas_categoria[question_id] = registro

        if len(preguntas_categoria) >= RAW_TARGET_PER_CATEGORY_STACKEXCHANGE:
            break

    ids_globales.update(preguntas_categoria.keys())
    df_categoria = convertir_a_dataframe(list(preguntas_categoria.values()), categoria)
    df_categoria['categoria_equipo'] = categoria
    todos_los_dataframes.append(df_categoria)
    print(f"Registros crudos únicos para {categoria}: {len(df_categoria)}")

print(f"\n✅ Listo. Se extrajeron las {len(todos_los_dataframes)} categorías.")


=== Extrayendo categoría: Backend ===
Consulta 'backend', página 1: 100 preguntas traídas.
Consulta 'backend', página 2: 100 preguntas traídas.
Consulta 'server-side', página 1: 100 preguntas traídas.
Consulta 'server-side', página 2: 100 preguntas traídas.
Registros crudos únicos para Backend: 394

=== Extrayendo categoría: Frontend ===
Consulta 'frontend', página 1: 100 preguntas traídas.
Consulta 'frontend', página 2: 100 preguntas traídas.
Consulta 'front-end', página 1: 100 preguntas traídas.
Consulta 'front-end', página 2: 100 preguntas traídas.
Registros crudos únicos para Frontend: 367

=== Extrayendo categoría: Data Science ===
Consulta 'data science', página 1: 100 preguntas traídas.
Consulta 'data science', página 2: 100 preguntas traídas.
Consulta 'machine learning', página 1: 100 preguntas traídas.
Consulta 'machine learning', página 2: 100 preguntas traídas.
Registros crudos únicos para Data Science: 397

=== Extrayendo categoría: Cloud ===
Consulta 'cloud computing', pá

In [13]:
df_completo = pd.concat(todos_los_dataframes, ignore_index=True)
print("Total de filas juntando todo:", len(df_completo))
df_completo["categoria_equipo"].value_counts()

Total de filas juntando todo: 2710


categoria_equipo
Data Science      397
Backend           394
Cloud             394
DevOps            393
Mobile            389
Bases de Datos    376
Frontend          367
Name: count, dtype: int64

In [14]:
df_completo = df_completo.drop_duplicates(subset='id_pregunta').reset_index(drop=True)
print(f'Registros únicos por id de pregunta: {len(df_completo)}')
df_completo["categoria_equipo"].value_counts()

Registros únicos por id de pregunta: 2710


categoria_equipo
Data Science      397
Backend           394
Cloud             394
DevOps            393
Mobile            389
Bases de Datos    376
Frontend          367
Name: count, dtype: int64

## 1.4 Normalización de categorías y tipo de contenido

Stack Overflow es un foro técnico de preguntas y respuestas. Cada registro se clasifica como un `apunte` técnico y el dataset cubre las siete categorías del proyecto.

In [15]:
CATEGORIAS_OBJETIVO = list(CATEGORY_KEYWORDS.keys())

# Conserva exclusivamente las siete categorías asignadas al proyecto
df_completo = df_completo[df_completo['categoria_equipo'].isin(CATEGORIAS_OBJETIVO)].reset_index(drop=True)
df_completo = df_completo.rename(columns={'categoria_equipo': 'categoria'})

# Las preguntas y respuestas de foro se consideran apuntes técnicos
df_completo['tipo'] = 'apunte'

assert set(df_completo['categoria'].unique()).issubset(CATEGORIAS_OBJETIVO)
print('✅ Categorías limitadas al catálogo oficial del proyecto.')
print(df_completo['categoria'].value_counts())
print(f"Tipo de contenido asignado: {df_completo['tipo'].unique().tolist()}")

✅ Categorías limitadas al catálogo oficial del proyecto.
categoria
Data Science      397
Backend           394
Cloud             394
DevOps            393
Mobile            389
Bases de Datos    376
Frontend          367
Name: count, dtype: int64
Tipo de contenido asignado: ['apunte']


## 1.5 Persistencia del dataset crudo

In [16]:
RAW_FOLDER = CARPETA_CRUDOS
PROCESSED_FOLDER = CARPETA_PROCESADOS

os.makedirs(RAW_FOLDER, exist_ok=True)
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

print("Carpeta crudos:", os.path.exists(RAW_FOLDER))
print("Carpeta procesados:", os.path.exists(PROCESSED_FOLDER))

Carpeta crudos: True
Carpeta procesados: True


In [17]:
ruta_completa_cruda = f'{RAW_FOLDER}/stackexchange_categorias_crudo.csv'
df_completo.to_csv(ruta_completa_cruda, index=False)

print("Guardado en:", ruta_completa_cruda)
print("Total de filas:", len(df_completo))

Guardado en: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\crudos/stackexchange_categorias_crudo.csv
Total de filas: 2710


## 1.6 Limpieza inicial del texto

In [18]:
def clean_html_urls(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'http\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df_completo['titulo'] = df_completo['titulo'].apply(clean_html_urls)
df_completo['texto'] = df_completo['texto'].apply(clean_html_urls)

# Eliminar duplicados basados en el contenido del apunte
initial_count = len(df_completo)
df_completo = df_completo.drop_duplicates(subset='texto').reset_index(drop=True)
print(f"Duplicados eliminados: {initial_count - len(df_completo)} (Filas restantes: {len(df_completo)})")

# Filtro de calidad: mínimo 100 palabras en la columna 'texto'
initial_count = len(df_completo)
df_completo = df_completo[df_completo['texto'].str.split().str.len() >= 100].reset_index(drop=True)
print(f"Textos con menos de 100 palabras eliminados: {initial_count - len(df_completo)} (Filas restantes: {len(df_completo)})")

print('✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.')

Duplicados eliminados: 0 (Filas restantes: 2710)
Textos con menos de 100 palabras eliminados: 1044 (Filas restantes: 1666)
✅ Limpieza de HTML, URLs, duplicados y textos cortos completada.


## 1.7 Preselección de candidatos para traducción

In [19]:
TRANSLATION_BUFFER_STACKEXCHANGE = 15
CANDIDATE_TARGET_STACKEXCHANGE = TARGET_PER_CATEGORY_STACKEXCHANGE + TRANSLATION_BUFFER_STACKEXCHANGE

df_completo = pd.concat([
    group.sample(min(len(group), CANDIDATE_TARGET_STACKEXCHANGE), random_state=42)
    for _, group in df_completo.groupby('categoria')
]).reset_index(drop=True)

print(f'✅ Candidatos preseleccionados (Máximo {CANDIDATE_TARGET_STACKEXCHANGE} por categoría).')

✅ Candidatos preseleccionados (Máximo 65 por categoría).


## 1.8 Balanceo final (50 registros por categoría)

In [20]:
available_counts = df_completo['categoria'].value_counts().reindex(CATEGORIAS_OBJETIVO, fill_value=0)
shortages = available_counts[available_counts < TARGET_PER_CATEGORY_STACKEXCHANGE]

if not shortages.empty:
    missing = {category: TARGET_PER_CATEGORY_STACKEXCHANGE - int(count) for category, count in shortages.items()}
    raise ValueError(f'No hay suficientes registros traducidos para completar StackExchange: {missing}')

df_completo = pd.concat([
    df_completo[df_completo['categoria'] == category].sample(
        n=TARGET_PER_CATEGORY_STACKEXCHANGE, random_state=42, replace=False
    )
    for category in CATEGORIAS_OBJETIVO
]).reset_index(drop=True)

print('✅ StackExchange balanceado en 50 registros únicos por categoría.')
print(df_completo['categoria'].value_counts().reindex(CATEGORIAS_OBJETIVO))

✅ StackExchange balanceado en 50 registros únicos por categoría.
categoria
Backend           50
Frontend          50
Data Science      50
Cloud             50
DevOps            50
Bases de Datos    50
Mobile            50
Name: count, dtype: int64


## 1.9 Traducción y procesamiento NLP

In [21]:
tqdm.pandas()

translation_errors = []

def translate_to_spanish(text):
    try:
        return GoogleTranslator(source='en', target='es').translate(str(text)[:1500])
    except Exception as e:
        translation_errors.append(type(e).__name__)
        return ""

# Traduce título y texto principal al español
df_completo['titulo_es'] = df_completo['titulo'].progress_apply(translate_to_spanish)
df_completo['texto_es'] = df_completo['texto'].progress_apply(translate_to_spanish)

if translation_errors:
    print(f"⚠️ {len(translation_errors)} traducciones fallaron. Tipos de error: {Counter(translation_errors)}")

def clean_nlp(text):
    # Quita signos de puntuación y pasa a minúsculas
    text = re.sub(r'[^\w\sáéíóúñ]', ' ', str(text).lower())
    # Elimina stopwords en español y palabras muy cortas
    return ' '.join([word for word in text.split() if word not in spanish_stopwords and len(word) > 2])

df_completo['texto_limpio'] = df_completo['texto_es'].apply(clean_nlp)

# Elimina filas donde la traducción del texto falló
initial_count = len(df_completo)
df_completo = df_completo[df_completo['texto_es'].str.strip() != ''].reset_index(drop=True)
print(f"\nTraducciones fallidas eliminadas: {initial_count - len(df_completo)} (Filas restantes: {len(df_completo)})")

# Guarda un respaldo antes de consolidar el esquema final
df_completo.to_csv(f'{CARPETA_PROCESADOS}/translation_backup_stackexchange.csv', index=False)
print('✅ Traducción y procesamiento NLP completados.')

# Reemplaza las columnas originales y descarta las auxiliares en la exportación final
df_completo['titulo'] = df_completo['titulo_es']
df_completo['texto'] = df_completo['texto_es']

  0%|          | 0/350 [00:00<?, ?it/s]

  0%|          | 0/350 [00:00<?, ?it/s]


Traducciones fallidas eliminadas: 0 (Filas restantes: 350)
✅ Traducción y procesamiento NLP completados.


## 1.10 Exportación final y auditoría de calidad

In [22]:
final_df_stackexchange = df_completo.copy()

# Estructura final: titulo, texto, categoria, autor, tipo
final_df_stackexchange = final_df_stackexchange[['titulo', 'texto', 'categoria', 'autor', 'tipo']]

assert final_df_stackexchange.columns.tolist() == ['titulo', 'texto', 'categoria', 'autor', 'tipo']
assert set(final_df_stackexchange['categoria'].unique()) == set(CATEGORIAS_OBJETIVO)

print('=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===')
print('   === DATASET - STACK OVERFLOW ===')
category_counts = final_df_stackexchange['categoria'].value_counts()
print(category_counts)

assert (category_counts.reindex(CATEGORIAS_OBJETIVO) == TARGET_PER_CATEGORY_STACKEXCHANGE).all()
print(f'\n✅ Todas las categorías cumplen {TARGET_PER_CATEGORY_STACKEXCHANGE} registros.')

output_path = f'{CARPETA_PROCESADOS}/dataset_FINAL_stackexchange.csv'
final_df_stackexchange.to_csv(output_path, index=False)
print(f'\n✅ Pipeline completado. Dataset exportado en: {output_path}')

print('\n=== MUESTRA DE AUDITORÍA (10 registros aleatorios) ===')
display(final_df_stackexchange.sample(min(10, len(final_df_stackexchange)), random_state=42))

=== DISTRIBUCIÓN FINAL POR CATEGORÍA ===
   === DATASET - STACK OVERFLOW ===
categoria
Backend           50
Frontend          50
Data Science      50
Cloud             50
DevOps            50
Bases de Datos    50
Mobile            50
Name: count, dtype: int64

✅ Todas las categorías cumplen 50 registros.

✅ Pipeline completado. Dataset exportado en: C:\Users\neo_p\Documents\G9-LATAM-Team-25\data_science\data\procesados/dataset_FINAL_stackexchange.csv

=== MUESTRA DE AUDITORÍA (10 registros aleatorios) ===


,titulo,texto,categoria,autor,tipo
157,¿Cómo generar una URL firmada por Blob en Goog...,"En Google Cloud Run, puede seleccionar qué cue...",Cloud,sww314,apunte
341,¿Por qué el emulador de Android es tan lento? ...,"Tengo un procesador Celeron de 2,67 GHz y 1,21...",Mobile,Andrie,apunte
315,"¿Apple rechaza el ""shell web móvil""? aplicacio...",No estoy seguro de cómo redactar esto correcta...,Mobile,danh32,apunte
234,Configuraciones específicas de la máquina web....,Tenemos varios equipos de desarrolladores en d...,DevOps,Tao,apunte
155,Ejecutar un comando Terraform a través del ope...,Estoy ejecutando Apache Airflow en Cloud Compo...,Cloud,parakeet,apunte
274,¿Es mejor utilizar varias bases de datos con u...,Después de este comentario a una de mis pregun...,Bases de Datos,Strae,apunte
304,Infracción invariante: no se pudo encontrar &q...,"Código completo aquí: Hola, tengo una aplicaci...",Mobile,user6015171,apunte
227,¿Cómo crear un entorno de desarrollo local par...,Kubernetes parece centrarse principalmente en ...,DevOps,Wernight,apunte
278,Explicación de la terminología BASE,El acrónimo BASE se utiliza para describir las...,Bases de Datos,Niels van der Rest,apunte
185,Función de nube para el tiempo de espera de Fi...,Estoy usando Firebase para una aplicación de c...,Cloud,Varun Gupta,apunte
